# Descriptors Computation
This notebook handles the descriptors computation. 
This procedure involves the following steps:
- For each variable, we estimate its Markov Blanket (MB) by selecting its lagged versions from one time step before and one after. 
- We standardize the time series to avoid varsortability
- Using the estimated MB, we compute a set of descriptors for all possible causal pairs (i.e., $ t-\tau \rightarrow t, \forall \tau$). that characterize the causal relationship between the variable pairs. These descriptors include conditional mutual information terms and other statistical properties that provide insights into the dependencies and interactions between the variables.
- For families of descriptors, we compute the quantiles of their empirical distributions. This step captures the distributional characteristics and aids in feature representation for the classifier.
- The computed descriptors and their quantiles are compiled into an input feature vector. This vector encapsulates the essential characteristics of the causal relationships and serves as the input for the classifier.
- For training data, each input vector is labeled as causal (1) or noncausal (0) based on the original selection criteria from the synthetic data's Directed Acyclic Graph (DAG). This labeling is crucial for supervised learning and model training.
- The labeled dataset, comprising the feature vectors, is used to train a classifier. The classifier learns to predict the likelihood of causal relationships based on the descriptors.
- For unseen time series data, the trained classifier predicts the probability of causal links for each pair of variables. The predictions are based on the computed descriptors for the test data.

Here are the descriptors 
\begin{align}
n  \\
m \\
m/n \\
b: \mathbf{z}_j = b \cdot (\mathbf{z}_i \oplus \mathbf{MB}_j) \text{, where} \oplus\text{indicates vector concatenation}  \\
b: \mathbf{z}_i = b \cdot (\mathbf{z}_j \oplus \mathbf{MB}_i) \text{, where} \oplus\text{indicates vector concatenation}   \\
\operatorname{kurt}(\mathbf{z}_i) = \mathbb{E}[(\mathbf{z}_i-\mathbb{E}[\mathbf{z}_i])^4] / \left(\mathbb{E}[(\mathbf{z}_i-\mathbb{E}[\mathbf{z}_i])^2]\right)^2 - 3 \\ 
\operatorname{kurt}(\mathbf{z}_j) = \mathbb{E}[(\mathbf{z}_j-\mathbb{E}[\mathbf{z}_j])^4] / \left(\mathbb{E}[(\mathbf{z}_j-\mathbb{E}[\mathbf{z}_j])^2]\right)^2 - 3 \\ 
\operatorname{skew}(\mathbf{z}_i) = \mathbb{E}[(\mathbf{z}_i-\mathbb{E}[\mathbf{z}_i])^3] / \left(\mathbb{E}[(\mathbf{z}_i-\mathbb{E}[\mathbf{z}_i])^2]\right)^{3/2} \\
\operatorname{skew}(\mathbf{z}_j) = \mathbb{E}[(\mathbf{z}_j-\mathbb{E}[\mathbf{z}_j])^3] / \left(\mathbb{E}[(\mathbf{z}_j-\mathbb{E}[\mathbf{z}_j])^2]\right)^{3/2}
 \\
\operatorname{HOC_{1,2}}(\mathbf{z}_i, \mathbf{z}_j) = \mathbb{E}\left[ (\mathbf{z}_i - \mathbb{E}[\mathbf{z}_i])^1 \cdot (\mathbf{z}_j - \mathbb{E}[\mathbf{z}_j])^2 \right] \\
\operatorname{HOC_{2,1}}(\mathbf{z}_i, \mathbf{z}_j) = \mathbb{E}\left[ (\mathbf{z}_i - \mathbb{E}[\mathbf{z}_i])^2 \cdot (\mathbf{z}_j - \mathbb{E}[\mathbf{z}_j])^1 \right] \\
\operatorname{HOC_{1,3}}(\mathbf{z}_i, \mathbf{z}_j) = \mathbb{E}\left[ (\mathbf{z}_i - \mathbb{E}[\mathbf{z}_i])^1 \cdot (\mathbf{z}_j - \mathbb{E}[\mathbf{z}_j])^3 \right] \\
\operatorname{HOC_{3,1}}(\mathbf{z}_i, \mathbf{z}_j) = \mathbb{E}\left[ (\mathbf{z}_i - \mathbb{E}[\mathbf{z}_i])^3 \cdot (\mathbf{z}_j - \mathbb{E}[\mathbf{z}_j])^1 \right] \\
    I(\mathbf{z}_i ; \mathbf{z}_j)  \\
    I(\mathbf{z}_j ; \mathbf{z}_i)  \\ 
    I(\mathbf{m}_j^{(k)} ; \mathbf{z}_i) \forall \mathbf{m}_j^{(k)} \in \mathbf{MB}_j  \\
    I(\mathbf{m}_i^{(k)} ; \mathbf{z}_j) \forall \mathbf{m}_i^{(k)} \in \mathbf{MB}_i \\
    I(\mathbf{z}_i ; \mathbf{z}_j | \mathbf{MB}_i \cap \mathbf{MB}_j)  \\
    I(\mathbf{z}_j ; \mathbf{z}_i | \mathbf{MB}_j)  \\
    I(\mathbf{z}_i ; \mathbf{z}_j | \mathbf{MB}_i)  \\
    I(\mathbf{z}_j ; \mathbf{z}_i | \mathbf{MB}_i \cup \mathbf{m}_j^{(k)}) \forall \mathbf{m}_j^{(k)} \in \mathbf{MB}_j  \\
    I(\mathbf{z}_i ; \mathbf{z}_j | \mathbf{MB}_j \cup \mathbf{m}_i^{(k)}) \forall \mathbf{m}_i^{(k)} \in \mathbf{MB}_i \\
    I(\mathbf{z}_i ; \mathbf{m}_j^{(k)} | \mathbf{z}_j) \forall \mathbf{m}_j^{(k)} \in \mathbf{MB}_j  \\
I(\mathbf{z}_j ; \mathbf{m}_i^{(k)} | \mathbf{z}_i) \forall \mathbf{m}_i^{(k)} \in \mathbf{MB}_i  \\
    I(\mathbf{m}_i^{(k_i)} ; \mathbf{m}_j^{(k_j)} | \mathbf{z}_i) \forall (\mathbf{m}_i^{(k_i)}, \mathbf{m}_j^{(k_j)}) \in \mathbf{MB_{i}} \times \mathbf{MB_{j}}  \\
    I(\mathbf{m}_j^{(k_j)} ; \mathbf{m}_i^{(k_i)} | \mathbf{z}_j) \forall (\mathbf{m}_i^{(k_i)}, \mathbf{m}_j^{(k_j)}) \in \mathbf{MB_{i}} \times \mathbf{MB_{j}} \\
        I(\mathbf{m}_i^{(k_{i}^{1})} ; \mathbf{m}_i^{(k_{i}^{2})} | \mathbf{z}_i) - I(\mathbf{m}_i^{(k_{i}^{1})} ; \mathbf{m}_i^{(k_{i}^{2})}) \forall (\mathbf{m}_i^{(k_{i}^{1})}, \mathbf{m}_i^{(k_{i}^{2})}) \in \mathbf{MB_{i}} \times \mathbf{MB_{i}}  \\
    I(\mathbf{m}_j^{(k_{j}^{1})} ; \mathbf{m}_j^{(k_{j}^{2})} | \mathbf{z}_j) - I(\mathbf{m}_j^{(k_{j}^{1})} ; \mathbf{m}_j^{(k_{j}^{2})}) \forall (\mathbf{m}_j^{(k_{j}^{1})}, \mathbf{m}_j^{(k_{j}^{2})}) \in \mathbf{MB_{j}} \times \mathbf{MB_{j}}     
\end{align}

In [1]:
from d2c.descriptors import D2C, DataLoader

In [2]:
N_VARS = 5
MAXLAGS = 3
N_JOBS = 50

The DataLoader class for handling time series data and directed acyclic graphs (DAGs) to prepare for the following stage: the descriptors computation.
The preparation includes:
1. Creating lagged time series: This step involves generating lagged versions of the original time series data. Lagged time series help in capturing temporal dependencies and interactions between variables at different time steps.
2. Flattening the original dictionaries into coherent lists: The original data, which may be stored in nested dictionaries, is flattened into lists. This transformation ensures that the data is in a consistent and accessible format for further processing.
3. Renaming the nodes of the DAGs: The nodes of the Directed Acyclic Graphs (DAGs) are renamed to maintain consistency and clarity. This step is crucial for accurately representing the causal relationships between variables in the DAGs.


In [3]:
dataloaders = {}
original_observations_training = {} 
lagged_flattened_observations_training = {} 
flattened_dags_training = {} 

for error_dist in ['gaussian', 'uniform', 'laplace']:
    dataloader = DataLoader(n_variables = N_VARS,
                            maxlags = MAXLAGS)
    dataloader.from_pickle(f'data/training_data_{error_dist}.pkl')
    
    dataloaders[error_dist] = dataloader
    original_observations_training[error_dist] = dataloader.get_original_observations()
    lagged_flattened_observations_training[error_dist] = dataloader.get_observations()
    flattened_dags_training[error_dist] = dataloader.get_dags()



In [4]:
original_observations_list_training = []
for obs_list in original_observations_training.values():
    original_observations_list_training.extend(obs_list) 

lagged_flattened_observations_list_training = []
for obs_list in lagged_flattened_observations_training.values():
    lagged_flattened_observations_list_training.extend(obs_list)

flattened_dags_list_training = []
for dags_list in flattened_dags_training.values():
    flattened_dags_list_training.extend(dags_list)

We are now ready for the core of our methodology: the D2C method. <br>
This method starts from a list of observations and dags and computes the corresponding descritpors, storing them in a dataframe. <br>
The D2C class gets the following arguments: 
- `observations` (list): List of observations (pd.DataFrame) corresponding to each DAG.
- `dags` (list): List of directed acyclic graphs (DAGs) representing causal relationships.
- `couples_to_consider_per_dag` (int, optional): To speedup, one can consider only a limited number of possible couples of variables. Couples are chosen to respect a specific ratio of causal/noncausal. Therefore, a DAG must be available: it can only be used for training data. For testing purposes only, we recommend using `D2CWrapper` instead. Defaults to -1 (all couples).  
- `n_variables` (int, optional): Number of variables in the time series. Defaults to 3.
- `maxlags` (int, optional): Maximum number of lags in the time series. Defaults to 3.
- `seed` (int, optional): Random seed for reproducibility. Defaults to 42.
- `n_jobs` (int, optional): Number of parallel jobs to run. Defaults to 1.
 - `full` (bool, optional): computes the whole set of descriptors rather than just a subset. Defaults to True. 



In [5]:
d2c_new = D2C(observations=lagged_flattened_observations_list_training[:1000],
        dags=flattened_dags_list_training[:1000], 
        couples_to_consider_per_dag=5, 
        n_variables=N_VARS, 
        maxlags=MAXLAGS,
        seed=42,
        n_jobs=30,
        full=True,
        dynamic=True,
        mb_estimator='ts',
        )

d2c_new.initialize()

Computing Descriptors:   0%|          | 0/10 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
d2c_new.get_descriptors_df().to_pickle('data/descriptors_df_train_newdynamic2.pkl')

In [7]:
dataloaders = {}
original_observations_testing = {} 
lagged_flattened_observations_testing = {} 
flattened_dags_testing = {}
true_causal_dfs = {}

for error_dist in ['gaussian', 'uniform', 'laplace']:
    dataloader = DataLoader(n_variables = N_VARS,
                            maxlags = MAXLAGS)
    dataloader.from_pickle(f'data/testing_data_{error_dist}.pkl')
    
    dataloaders[error_dist] = dataloader
    original_observations_testing[error_dist] = dataloader.get_original_observations()
    lagged_flattened_observations_testing[error_dist] = dataloader.get_observations()
    flattened_dags_testing[error_dist] = dataloader.get_dags()
    true_causal_dfs[error_dist] = dataloader.get_true_causal_dfs()

original_observations_list_testing = []
for obs_list in original_observations_testing.values():
    original_observations_list_testing.extend(obs_list) 

lagged_flattened_observations_list_testing = []
for obs_list in lagged_flattened_observations_testing.values():
    lagged_flattened_observations_list_testing.extend(obs_list)

flattened_dags_list_testing = []
for dags_list in flattened_dags_testing.values():
    flattened_dags_list_testing.extend(dags_list)

true_causal_dfs_list_testing = []
for causal_df in true_causal_dfs.values():
    true_causal_dfs_list_testing.extend(causal_df)

In [8]:
from d2c.descriptors import D2C, DataLoader
d2c_new = D2C(observations=lagged_flattened_observations_list_testing,
        dags=flattened_dags_list_testing, 
        couples_to_consider_per_dag=5, 
        n_variables=N_VARS, 
        maxlags=MAXLAGS,
        seed=42,
        n_jobs=N_JOBS,
        full=True,
        dynamic=True,
        mb_estimator='ts',
        )

d2c_new.initialize()
d2c_new.get_descriptors_df().to_pickle('data/descriptors_test_newdynami2c.pkl')

Processing DAGs:   0%|          | 0/1080 [00:00<?, ?it/s]

/home/gpaldino/miniconda3/envs/td2c/lib/python3.8/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/gpaldino/miniconda3/envs/td2c/lib/python3.8/site-packages/numpy/lib/function_base.py:2855: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/gpaldino/miniconda3/envs/td2c/lib/python3.8/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/gpaldino/miniconda3/envs/td2c/lib/python3.8/site-packages/numpy/lib/function_base.py:2855: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/gpaldino/miniconda3/envs/td2c/lib/python3.8/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/gpaldino/miniconda3/envs/td2c/lib/python3.8/site-packages/numpy/lib/function_base.py:2855: RuntimeWarning: invalid value encountered i

We remind our variable naming convention: <br>
- A time series of `n_variables` dimensions will have names from 0 to `n_variables - 1` to refer to the variables at time `t` (present)
- names from `n_variables` to `n_variables*2 - 1` will indicate the same variable at time `t-1` (1-lag)
- names from `n_variables*2` to `n_variables*3 - 1` will indicate the same variable at time `t-2` (2-lag)  <br>
For example if `n_variables = 5`, the line where `edge_source` is 12 and `edge_dest` is 4, refers to the link between variable `3` at `t-2` to variable `5` at time `t`

In [16]:
from d2c.descriptors import DataLoader
N_JOBS = 50
MAXLAGS = 3
N_VARS = 5
dataloader = DataLoader(n_variables = N_VARS,
                            maxlags = MAXLAGS)
dataloader.from_pickle(f'realdata/netsym/netsym_5.pkl')

original_observations_testing= dataloader.get_original_observations()
lagged_flattened_observations_testing = dataloader.get_observations()
flattened_dags_testing= dataloader.get_dags()
true_causal_dfs = dataloader.get_true_causal_dfs()

In [20]:
from d2c.descriptors import D2C, DataLoader
d2c_new = D2C(observations=lagged_flattened_observations_list_testing,
        dags=flattened_dags_list_testing, 
        couples_to_consider_per_dag=-1, 
        n_variables=N_VARS, 
        maxlags=MAXLAGS,
        seed=42,
        n_jobs=20,
        full=True,
        dynamic=True,
        mb_estimator='ts',
        )

d2c_new.initialize()
d2c_new.get_descriptors_df().to_pickle('data/descriptors_netsim_5_full.pkl')

Processing DAGs:   0%|          | 0/1080 [00:00<?, ?it/s]